# Universe — step 2 of 8

**In plain words:** the eligible list, rebuilt for each date rather than for today.

**It produces** `Universe/Security_Master.csv` and `Universe/Data_Issues.csv`.

**It prevents** survivorship bias — testing on the winners that happened to survive.

> **Runs after `Data/curator.py`**, because it profiles the files that were downloaded, and
> **before `Data/refinery.py`**, because that stage joins the security master this one writes.

**The universe is decided in `Universe/` and nowhere else** — the seed says which securities, this
notebook what each one is and from when. Nothing downstream second-guesses either.

```
Universe/Investable_Universe.csv   the seed, committed  ->  edit this to change the universe
        |
        +--> Data/curator.py       downloads one file per identifier in it
        |
        +--> this notebook         Security_Master.csv, Data_Issues.csv
```

**In the template this notebook holds no code:** each section below says what is expected in it.
Write the cells.

<!-- example: begin -->

In this example the universe starts from the index. `Universe/seed.py` wrote the seed from
the KN US Equity Core's membership: every listing that was a member on some date from
2015-01-02, matched or not, with the FMP symbol its prices are filed under where one was
found. This notebook also writes
`Universe/Coverage.csv`, the share of the index's weight FMP prices on each date, and fixes
the date the experiment's window opens, `Universe/Window.csv`, before any rule exists.

<!-- example: end -->

## 0 · Setup

Paths and the provider key, read from `Config/.env`. Nothing here touches the network, and no
value from that file is ever printed.

## The seed

`Universe/Investable_Universe.csv` is the only thing that decides what this repository is about.
Replace its rows with equities, ETFs, FX crosses, crypto pairs or futures and every stage below
still runs — nothing downstream names an asset class.

**One column is required: `main_identifier`**, the identifier the experiment's provider prices
each security under, which the Data Curator asks it for. A seed taken from an index also carries
the index's own ticker, `index_identifier`, which can differ; the map between the two comes from
the strategy's seed builder, or from the Analytics Factory when it supplies one.
Every other column is yours. Most strategies want at least a readable name and whatever they group
securities by; add those columns here and they flow through the whole pipeline.

**A span ends where the security did, never where its source does.** A seed that carries the
dates each identifier speaks for — a first and a last, so a recycled symbol never joins two
companies — leaves the last empty for a listing still trading on its source's last date: that
is the day the source was written, not a delisting. Closed there, every live name stops on that
day; no backtest cut earlier can notice, and a paper book finds nothing to price the day after.

<!-- example: begin -->

`Universe/Investable_Universe.csv` holds **every listing that was a KN US Equity Core member on
some date from 2015-01-02**, matched or not. `main_identifier` is the FMP symbol its prices are
filed under — empty when FMP does not price it — `index_identifier` the index's own ticker,
`valid_from` and `valid_to` the span the symbol speaks for the security, and `match` how the two
were tied together. `Universe/seed.py` writes it; this notebook only ever drops a match whose
prices contradict it. FMP is this experiment's provider: a later experiment may use another,
with the main identifier KaxaNuk supplies for it. The only network calls here are FMP profiles
for symbols `Universe/seed.py` did not already cache.

<!-- example: end -->

In [ ]:
# EXAMPLE-ONLY CELL
import csv
import datetime
import json
import os
import pathlib
import sys
import time
import urllib.error
import urllib.parse
import urllib.request

import pandas

import kaxanuk.data_curator

# Every path below is relative to the repository root, whichever folder the notebook was started in.
NOTEBOOK_DIRECTORY = pathlib.Path.cwd()
REPOSITORY_ROOT = (
    NOTEBOOK_DIRECTORY
    if (NOTEBOOK_DIRECTORY / "Universe").is_dir()
    else NOTEBOOK_DIRECTORY.parent
)
os.chdir(REPOSITORY_ROOT)
# The index's files are the Analytics Factory's, read in place by one module, never renamed or
# re-headed.
sys.path.insert(0, str(REPOSITORY_ROOT / "Data"))

import hand_supplied  # noqa: E402 - the path above has to exist first

SEED_PATH = pathlib.Path("Universe/Investable_Universe.csv")
CURATOR_DIRECTORY = pathlib.Path("Data/Curator/Time_Series")
FETCH_OUTCOMES_PATH = pathlib.Path("Data/Curator/fetch_outcomes.json")
PROVIDER_CACHE_PATH = pathlib.Path("Universe/Provider_Cache/profiles.json")
SECURITY_MASTER_PATH = pathlib.Path("Universe/Security_Master.csv")
DATA_ISSUES_PATH = pathlib.Path("Universe/Data_Issues.csv")
COVERAGE_PATH = pathlib.Path("Universe/Coverage.csv")
WINDOW_PATH = pathlib.Path("Universe/Window.csv")
PRICE_COLUMN = "m_close_dividend_and_split_adjusted"

# The owner's floor and the experiment's last day: the window opens no earlier than the first, and
# everything after the second is unseen until a book is frozen.
WINDOW_FLOOR = pandas.Timestamp("2015-01-02")
PANEL_END = pandas.Timestamp("2026-06-01")
# The owner's rule of 2026-10-05: the window opens on the first date from which the members FMP
# prices hold at least this share of the index's weight on every date to the window's end.
COVERAGE_REQUIRED = 0.90
# The golden cross needs 200 trading days of prices before it can say anything.
WARM_UP_DAYS = 200
# The data checks look at what the rule can read: the window and the warm-up before it.
CHECK_FROM = WINDOW_FLOOR - pandas.Timedelta(450, unit="D")
# A price file that starts or ends this far outside the security's own span carries another
# company's history under the same symbol, and only the span is read.
SPAN_TOLERANCE_DAYS = 30
# An unverified match whose FMP prices cover less than this share of the days it was a member is
# not the same security, and is dropped.
UNVERIFIED_OVERLAP_REQUIRED = 0.5
IMPOSSIBLE_DAILY_MOVE = 5.0
# A rate limit or a dropped connection is a pause, not an answer: the profile is asked again after
# this many seconds, this many times, as `Universe/seed.py` asks.
REQUEST_ATTEMPTS = 4
THROTTLE_PAUSE_SECONDS = 30

# The key is loaded into the environment and read by name only; no value is printed anywhere here.
kaxanuk.data_curator.load_config_env()

seed_rows = pandas.read_csv(SEED_PATH, dtype=str, keep_default_na=False)
priced_rows = seed_rows[seed_rows["main_identifier"] != ""]
print(f"seed: {len(seed_rows)} index listings, {len(priced_rows)} with an FMP symbol")
print(seed_rows["match"].value_counts().to_string())

## 1 · The seed, checked

Two things to establish before anything downstream trusts this file.

1. **Every identifier is unique**; another identity column you carry, an ISIN, repeats only
   where a renamed security's rows follow each other in time. A repeated identity under two rows is usually a renamed security, and the two legs have to be stitched
   into one position or the book holds it twice. A point-in-time universe is supposed to contain
   these; what it must not do is hide them.
2. **The grouping is complete.** Whatever column the strategy compares things by, a missing value
   in it is a security that silently drops out of every group-level view.

In [ ]:
# EXAMPLE-ONLY CELL
# 1. Every FMP symbol once and every index listing once: two listings on one price file would hold
#    one company twice, and one listing on two would split its history.
duplicate_symbols = priced_rows["main_identifier"][priced_rows["main_identifier"].duplicated()]
duplicate_listings = seed_rows["index_identifier"][seed_rows["index_identifier"].duplicated()]
print(f"duplicate FMP symbols: {len(duplicate_symbols)}")
print(f"duplicate index listings: {len(duplicate_listings)}")

# 2. The listings FMP does not price, named: their weight is the survivorship cost of section 5.
unpriced_listings = seed_rows.loc[
    seed_rows["main_identifier"] == "",
    ["index_identifier", "name", "match"],
]
print(f"index listings with no FMP symbol: {len(unpriced_listings)}")
print(unpriced_listings.to_string(index=False))

## 2 · The security master

The seed gives identity. Everything else comes from the provider: the official name, what kind of
instrument it is, where it trades, in what currency, and when it started.

**Cache the provider's raw payload, then shape the master from the cache.** Re-running then costs
nothing, changing the column mapping never triggers a refetch, and the untouched payload stays
available for fields this notebook does not yet use. Adding a provider is one fetch function and
one normaliser.

**The seed wins.** Its identity columns define the universe, so a provider value never overwrites
one — it is compared instead, and a disagreement is reported. A provider that now points a symbol
at a different security is a recycled identifier, and joining on it across the whole history would
silently mix two companies.

> **What the master carries is what a security is *today*.** The provider keeps no history, so on
> a universe whose members get reclassified, every period before the move is attributed wrongly and
> nothing raises an error. That is why the refinery prefixes every joined column `current_`.

In [ ]:
# EXAMPLE-ONLY CELL
# The provider is asked once per symbol and the raw payload is cached under the symbol asked for,
# as `Universe/seed.py` caches it, so re-running this notebook costs nothing.
PROVIDER_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
cached_profiles = (
    json.loads(PROVIDER_CACHE_PATH.read_text(encoding="utf-8"))
    if PROVIDER_CACHE_PATH.is_file()
    else {}
)
identifiers = list(priced_rows["main_identifier"])
pending = [identifier for identifier in identifiers if identifier not in cached_profiles]
print(f"cached: {len(cached_profiles)}, to fetch: {len(pending)}")

for position, identifier in enumerate(pending, start=1):
    request_url = (
        "https://financialmodelingprep.com/stable/profile?symbol="
        + urllib.parse.quote(identifier)
        + "&apikey="
        + os.environ["KNDC_API_KEY_FMP"]
    )

    for attempt in range(1, REQUEST_ATTEMPTS + 1):
        try:
            with urllib.request.urlopen(request_url, timeout=60) as response:
                payload = json.loads(response.read().decode("utf-8"))

            break
        except urllib.error.URLError as error:
            # HTTP 429, or no connection at all, is asked again; any other refusal stops the run.
            throttled = not isinstance(error, urllib.error.HTTPError) or error.code == 429

            if not throttled or attempt == REQUEST_ATTEMPTS:

                raise

        print(f"{identifier}: no answer on attempt {attempt}; pausing", flush=True)
        time.sleep(THROTTLE_PAUSE_SECONDS)

    cached_profiles[identifier] = payload[0] if len(payload) > 0 else {}

    if position % 50 == 0:
        PROVIDER_CACHE_PATH.write_text(json.dumps(cached_profiles, indent=1), encoding="utf-8")

PROVIDER_CACHE_PATH.write_text(json.dumps(cached_profiles, indent=1), encoding="utf-8")
print(f"payloads cached: {len(cached_profiles)}")

In [ ]:
# EXAMPLE-ONLY CELL
# The seed wins on identity: the index's name and span are kept, the provider adds what it knows.
master_rows = []

for row in priced_rows.itertuples(index=False):
    profile = cached_profiles.get(row.main_identifier) or {}
    master_rows.append({
        "main_identifier": row.main_identifier,
        "index_identifier": row.index_identifier,
        "name": row.name or profile.get("companyName"),
        "instrument_type": "etf" if profile.get("isEtf") else "equity",
        "exchange": profile.get("exchange"),
        "currency": profile.get("currency"),
        "inception": profile.get("ipoDate"),
        "isin": profile.get("isin") or row.isin or None,
        "is_actively_trading": profile.get("isActivelyTrading"),
        # Today's classification, never point-in-time: the refinery joins these as `current_*`.
        "sector": profile.get("sector"),
        "industry": profile.get("industry"),
        "valid_from": row.valid_from or None,
        "valid_to": row.valid_to or None,
        "match": row.match,
    })

security_master = pandas.DataFrame(master_rows)
security_master.to_csv(SECURITY_MASTER_PATH, index=False)
print(f"wrote {SECURITY_MASTER_PATH}: {len(security_master)} rows")
print(f"symbols the provider has no profile for: {int(security_master['exchange'].isna().sum())}")
security_master.head()

## 3 · Composition, and how long each member has existed

Two facts decide what a backtest can honestly claim. **What the universe is made of** sets what
diversification is even available; **when each member started trading** sets the window, because a
universe is only complete from the inception of its youngest member.

## 4 · The data-issues register

The seed says what *should* exist. This section reads what *does*, and writes
`Universe/Data_Issues.csv`. Every check below has cost somebody real time somewhere:

| Check | The failure it catches |
| --- | --- |
| **Missing file** | an identifier the provider does not carry, which becomes a silent hole in the panel |
| **Schema drift** | a folder holding two column sets, which makes every downstream read conditional |
| **No usable signal** | a history shorter than the strategy's longest warm-up, so the name can never be selected |
| **Unusable values** | zero or negative prices, which break every return calculation downstream |
| **Impossible daily move** | an adjusted price that multiplies by more than six in a day: an unadjusted corporate action or a bad print, not a return |
| **Late start** | a series that begins after the panel does, so the cross-section is smaller before that date |
| **Status disagreement** | a series that ends early on a name the provider still calls active: a data gap, not a delisting |
| **Early end** | a series that stops early — delisted or halted, and a held position must be exited on its last priced day |

**Each row has a severity, blocking first.** A blocking row lists names no book may hold: every
experiment reads them from this file and excludes them by name, so the row is never just a label.

The `universe-point-in-time` skill lists three more: **internal gaps**, **identity conflict** and
**traded value broken at a split**. Identity conflict needs an identity column beyond
`main_identifier`: a provider cache keyed by the symbol the provider returns cannot disagree with
the seed on that symbol alone. The split check is for any strategy that ranks, weights or screens
on traded value: at every split, the median traded value of the thirty trading days after, against
the thirty before, stays near one on a correct series, and lands near the split factor where the
provider's volume counted the split twice.

<!-- example: begin -->

*This check was added after the record (issue #14); this copy's run did not perform it. Its
register lists its rows in the order its checks run, not blocking first.*

Here the register holds eleven checks. Beside the rows above: **No FMP symbol**, an index
member this repository cannot price, the survivorship cost by name; and **Spliced**, a file
that runs outside its security's own span — another company under a reused symbol — masked
to the span. **Missing file** counts a symbol FMP does not carry apart from one it refused
tonight, because only the second is worth asking again.

<!-- example: end -->

In [ ]:
# EXAMPLE-ONLY CELL
# What the universe is made of, and how much of it is already dead. A seed showing 0% delisted
# would not be a universe, it would be a survivor list.
valid_to = pandas.to_datetime(security_master["valid_to"], errors="coerce")
ended = valid_to < PANEL_END
print(f"prices end before {PANEL_END.date()}: {int(ended.sum())} of {len(security_master)}")
print(security_master["sector"].value_counts(dropna=False).head(15).to_string())
print(security_master["instrument_type"].value_counts(dropna=False).to_string())

In [ ]:
# EXAMPLE-ONLY CELL
# The checks run against what the curator actually wrote, inside each security's span, over the
# window and its warm-up. Each row names the identifiers, so the next stage gets a work list.
fetch_outcomes = (
    json.loads(FETCH_OUTCOMES_PATH.read_text(encoding="utf-8"))
    if FETCH_OUTCOMES_PATH.is_file()
    else {}
)
spans = {
    row.main_identifier: (
        pandas.Timestamp(row.valid_from) if row.valid_from else None,
        pandas.Timestamp(row.valid_to) if row.valid_to else None,
    )
    for row in priced_rows.itertuples(index=False)
}

price_frames = {}
short_history = []
non_positive_prices = []
impossible_moves = []
spliced = []
expected_header = None
schema_drift = []

for identifier in identifiers:
    path = CURATOR_DIRECTORY / f"{identifier}.csv"

    if not path.is_file():
        continue

    with path.open(encoding="utf-8-sig", newline="") as handle:
        header = tuple(next(csv.reader(handle), []))

    if expected_header is None:
        expected_header = header
    elif header != expected_header:
        schema_drift.append(identifier)

    whole = pandas.read_csv(
        path,
        usecols=["m_date", PRICE_COLUMN],
        parse_dates=["m_date"],
    )
    span_start, span_end = spans[identifier]
    tolerance = pandas.Timedelta(SPAN_TOLERANCE_DAYS, unit="D")
    starts_early = span_start is not None and whole["m_date"].min() < span_start - tolerance
    ends_late = span_end is not None and whole["m_date"].max() > span_end + tolerance

    if starts_early or ends_late:
        spliced.append(identifier)

    inside = pandas.Series(True, index=whole.index)

    if span_start is not None:
        inside &= whole["m_date"] >= span_start

    if span_end is not None:
        inside &= whole["m_date"] <= span_end

    frame = whole.loc[inside].reset_index(drop=True)
    price_frames[identifier] = frame

    if len(frame) < WARM_UP_DAYS:
        short_history.append(identifier)

    checked = frame[(frame["m_date"] >= CHECK_FROM) & (frame["m_date"] <= PANEL_END)]

    if (checked[PRICE_COLUMN] <= 0).any():
        non_positive_prices.append(identifier)

    # A price that multiplies by six in one day is an unadjusted corporate action or a bad print,
    # not a return. Real squeezes reach three or four times; the threshold sits above them on
    # purpose, so a flag here means the series is wrong rather than merely wild.
    daily_move = checked[PRICE_COLUMN].pct_change(fill_method=None)

    if (daily_move.abs() > IMPOSSIBLE_DAILY_MOVE).any():
        impossible_moves.append(identifier)

missing_files = [
    identifier
    for identifier in identifiers
    if identifier not in price_frames
]
# FMP's own answer separates a name it does not carry from one it refused tonight.
not_carried = [
    identifier
    for identifier in missing_files
    if fetch_outcomes.get(identifier) in ("not carried", "unusable")
]
refused = [
    identifier
    for identifier in missing_files
    if identifier not in not_carried
]
late_start = [
    identifier
    for identifier, frame in price_frames.items()
    if len(frame) > 0 and frame["m_date"].min() > WINDOW_FLOOR
]
early_end = [
    identifier
    for identifier, frame in price_frames.items()
    if len(frame) > 0 and frame["m_date"].max() < PANEL_END - datetime.timedelta(days=7)
]
# A series that stops while the provider still calls the security active is a data gap, not a
# delisting -- and the two need different answers, so they are counted apart.
active = set(security_master.loc[security_master["is_actively_trading"] == True, "main_identifier"])  # noqa: E712
status_disagreement = [
    identifier
    for identifier in early_end
    if identifier in active
]
print(f"files read: {len(price_frames)}, missing: {len(missing_files)} "
      f"({len(not_carried)} not carried by FMP, {len(refused)} refused or never asked)")

In [ ]:
# EXAMPLE-ONLY CELL
# An unverified match -- a dead listing FMP keeps no profile for, under the index's own ticker -- is
# kept only if FMP's prices cover most of the days the listing was a member. Otherwise the file is
# another company's, and the listing joins the ones FMP does not price.
holdings = hand_supplied.read_benchmark_holdings()
member_days = (holdings > 0).sum(axis=0)
contradicted = []

for row in priced_rows.itertuples(index=False):
    if row.match != "unverified":
        continue

    frame = price_frames.get(row.main_identifier)
    membership = holdings.index[holdings[row.index_identifier] > 0]

    if frame is None or len(membership) == 0:
        continue

    priced_dates = set(frame["m_date"])
    overlap = sum(date in priced_dates for date in membership) / len(membership)

    if overlap < UNVERIFIED_OVERLAP_REQUIRED:
        contradicted.append(row.main_identifier)

if len(contradicted) > 0:
    dropped = seed_rows["main_identifier"].isin(contradicted)
    seed_rows.loc[dropped, "match"] = "unverified, FMP dates disagree"
    seed_rows.loc[dropped, "main_identifier"] = ""
    seed_rows.to_csv(SEED_PATH, index=False, lineterminator="\n")

    for identifier in contradicted:
        price_frames.pop(identifier, None)

    security_master = security_master[~security_master["main_identifier"].isin(contradicted)]
    security_master.to_csv(SECURITY_MASTER_PATH, index=False)

print(f"unverified matches dropped for dates that disagree: {len(contradicted)} {contradicted}")

In [ ]:
# EXAMPLE-ONLY CELL
unpriced_now = seed_rows.loc[seed_rows["main_identifier"] == "", "index_identifier"].tolist()
issues = pandas.DataFrame([
    {
        "check": "no FMP symbol (index listing FMP does not price)",
        "severity": "survivorship",
        "count": len(unpriced_now),
        "identifiers": " ".join(unpriced_now),
    },
    {
        "check": "missing file (FMP does not carry the symbol)",
        "severity": "survivorship",
        "count": len(not_carried),
        "identifiers": " ".join(not_carried),
    },
    {
        "check": "missing file (refused by the provider, or never asked)",
        "severity": "blocking",
        "count": len(refused),
        "identifiers": " ".join(refused),
    },
    {
        "check": "schema drift",
        "severity": "blocking",
        "count": len(schema_drift),
        "identifiers": " ".join(schema_drift),
    },
    {
        "check": "no usable signal (history shorter than the warm-up)",
        "severity": "blocking",
        "count": len(short_history),
        "identifiers": " ".join(short_history),
    },
    {
        "check": "unusable values (zero or negative prices)",
        "severity": "blocking",
        "count": len(non_positive_prices),
        "identifiers": " ".join(non_positive_prices),
    },
    {
        "check": "impossible daily move (the adjusted price multiplies by more than six)",
        "severity": "blocking",
        "count": len(impossible_moves),
        "identifiers": " ".join(impossible_moves),
    },
    {
        "check": "spliced (the file runs outside the security's own span; only the span is read)",
        "severity": "masked",
        "count": len(spliced),
        "identifiers": " ".join(spliced),
    },
    {
        "check": "late start (no prices when the floor opens in 2015)",
        "severity": "expected",
        "count": len(late_start),
        "identifiers": " ".join(late_start),
    },
    {
        "check": "status disagreement (series ends early, provider still calls it active)",
        "severity": "blocking",
        "count": len(status_disagreement),
        "identifiers": " ".join(status_disagreement),
    },
    {
        "check": "early end (last price before the panel ends: delisted, acquired or halted)",
        "severity": "expected",
        "count": len(early_end),
        "identifiers": " ".join(early_end),
    },
])
issues.to_csv(DATA_ISSUES_PATH, index=False)
print(f"wrote {DATA_ISSUES_PATH}")
issues[["check", "severity", "count"]]

## 5 · When the universe is actually usable

A file that starts in 2010 gives no signal in 2010. Every feature has a warm-up, and a five-year
one moves the honest start of a backtest by five years.

**The date that matters is the first day on which the universe can be both priced and
signalled** — every security in it, or the share of it the strategy declares, with the shortfall
published beside every result. A shortfall is a member the provider does not carry, never a listing
whose key did not join. Before that date the strategy is choosing from a smaller menu than it
appears to be, and a backtest that starts earlier is quietly comparing books drawn from different
universes. Nothing else in the pipeline says so, which is why it is answered here.

<!-- example: begin -->

Here the window opens on the later of the floor, 2015-01-02, and the first date from which the
members with an FMP price on that date hold at least 90% of the index's weight on every date
to the window's end — the owner's rule of 2026-10-05. The coverage is written per date to
`Universe/Coverage.csv`, and the share the index loses — the members FMP does not carry, and
the ones outside their file's span — is **the survivorship cost of an FMP-only universe**,
quoted beside every result. It understates the cost for this rule: the names missing are
disproportionately large dead companies, and the rule ranks by traded value.

<!-- example: end -->

In [ ]:
# EXAMPLE-ONLY CELL
keys = hand_supplied.read_listing_keys()
window_dates = holdings.index[(holdings.index >= WINDOW_FLOOR) & (holdings.index <= PANEL_END)]
on_window = holdings.loc[window_dates]
priced_matrix = pandas.DataFrame(False, index=window_dates, columns=holdings.columns)

for listing in holdings.columns:
    symbol = keys.get(listing)

    if symbol is None or symbol not in price_frames:
        continue

    series = price_frames[symbol].set_index("m_date")[PRICE_COLUMN]
    priced_matrix[listing] = series.reindex(window_dates).notna().to_numpy()

members = on_window > 0
index_weight = on_window.sum(axis=1)
priced_weight = on_window.where(priced_matrix, 0.0).sum(axis=1)
coverage_share = priced_weight / index_weight
coverage = pandas.DataFrame({
    "members": members.sum(axis=1),
    "members_priced": (members & priced_matrix).sum(axis=1),
    "index_weight": index_weight.round(6),
    "priced_weight": priced_weight.round(6),
    "coverage": coverage_share.round(6),
})
coverage.index.name = "m_date"
coverage.to_csv(COVERAGE_PATH)

short_of_required = coverage_share[coverage_share < COVERAGE_REQUIRED]
window_start = (
    window_dates[window_dates > short_of_required.index.max()].min()
    if len(short_of_required) > 0
    else window_dates.min()
)
print(f"coverage on the floor, {WINDOW_FLOOR.date()}: {coverage_share.iloc[0]:.2%}")
print(f"dates below {COVERAGE_REQUIRED:.0%}: {len(short_of_required)}")
print(f"THE WINDOW OPENS ON {window_start.date()} and closes on {PANEL_END.date()}")
# Written for the analyzer, which screens over the same window; the experiment states it in its
# blueprint and checks it against this file.
pandas.DataFrame([{
    "window_start": window_start.date().isoformat(),
    "window_end": PANEL_END.date().isoformat(),
    "coverage_required": COVERAGE_REQUIRED,
    "coverage_on_floor": round(float(coverage_share.iloc[0]), 6),
}]).to_csv(WINDOW_PATH, index=False)
by_year = coverage.groupby(coverage.index.year).agg(
    members=("members", "mean"),
    members_priced=("members_priced", "mean"),
    coverage_mean=("coverage", "mean"),
    coverage_min=("coverage", "min"),
)
print(by_year.round(3).to_string())

## 6 · Handoff

| Output | Consumed by |
| --- | --- |
| `Universe/Security_Master.csv` | `Data/refinery.py`, which joins its columns onto the panel |
| `Universe/Data_Issues.csv` | the caveats section of every `FINDINGS_N.md` |

## Open items to carry forward

| # | Item | Why it matters |
| --- | --- | --- |
| 1 | **Classification is a snapshot, not a history.** | A security reclassified mid-window is misattributed before its move. The `current_*` prefix marks exactly this. |
| 2 | **The last day of a delisted name is unaudited.** Nothing here checks whether a truncated series ends on a real final price or on a provider gap. | On a universe that retains delisted names, the missing returns are disproportionately the bad ones. |
| 3 | **Inception dates come from the provider, not from the price file.** | The price file is what the backtest actually trades; the master is what a reader believes. Where they disagree, say so. |

<!-- example: begin -->

Here two more files hand off: `Universe/Coverage.csv` to the survivorship row of `RESULTS.md`
and `FINDINGS_1.md`, and `Universe/Window.csv` to `Data/analyzer.ipynb`, which screens over the
same window; the window's first date goes to `JOURNAL_1.md`, then `BLUEPRINT_1.md`, before any
rule exists. The register's blocking rows are the experiment's exclusions. Open here: **the
members FMP does not carry are dropped, not filled** — the owner's decision of 2026-10-05.
Their weight is published per date, and it is concentrated in the large dead companies a
traded-value rule would have held.

<!-- example: end -->

In [ ]:
# EXAMPLE-ONLY CELL
print(f"{SECURITY_MASTER_PATH}: {len(security_master)} rows -> Data/refinery.py joins as current_*")
print(f"{DATA_ISSUES_PATH}: {len(issues)} checks -> the caveats table of FINDINGS_1.md")
print(f"{COVERAGE_PATH}: {len(coverage)} dates -> the survivorship row of RESULTS.md")
print(f"the window: {window_start.date()} to {PANEL_END.date()} -> JOURNAL_1.md, BLUEPRINT_1.md")

## 7 · Verify

**Assertions that raise when this stage's output is wrong**, read back from the files this notebook
wrote rather than from the variables that wrote them. A check that prints is one somebody has to
read; one that raises stops the run, so a notebook that reaches its last cell is one whose outputs
hold. At the least:

- `Universe/Security_Master.csv` has one row per identifier in the seed, each once, and no
  identifier the seed lacks;
- `Universe/Data_Issues.csv` has a row for every check section 4 runs, each with a count;
- the downloaded files and the register's missing ones account for the whole seed;
- section 5 found a date from which the universe is usable;
- for a seed taken from an index: every listing the index weights from the window's floor, the
  earliest date it may open, is a row of the seed under `index_identifier`; every row with no
  `main_identifier` is named in the register; no `main_identifier` or `index_identifier` appears
  twice; and no two rows hold one ISIN over the same dates — a renamed security's two rows follow
  each other.

<!-- example: begin -->

Here the Verify section also checks that from the window's first date to its last the members
FMP prices hold at least 90% of the index's weight. The ISIN check was added after the record
(issue #13), and this copy's Verify does not run it; its seed holds no ISIN twice.

<!-- example: end -->

In [ ]:
# EXAMPLE-ONLY CELL
# Read back what this notebook wrote, and raise on the first thing that is wrong. A printed PASS is
# read by whoever is watching; a raised error stops a run nobody is.
written_seed = pandas.read_csv(SEED_PATH, dtype=str, keep_default_na=False)
written_priced = written_seed[written_seed["main_identifier"] != ""]
written_master = pandas.read_csv(SECURITY_MASTER_PATH)
written_issues = pandas.read_csv(DATA_ISSUES_PATH).set_index("check")
written_coverage = pandas.read_csv(COVERAGE_PATH, parse_dates=["m_date"]).set_index("m_date")
missing_count = int(written_issues["count"].filter(like="missing file").sum())
since_floor = holdings.loc[holdings.index >= WINDOW_FLOOR]
weighted_listings = set(since_floor.columns[(since_floor > 0).any(axis=0)])
unpriced_row = written_issues.loc["no FMP symbol (index listing FMP does not price)"]
named_unpriced = set(str(unpriced_row["identifiers"]).split())
unpriced_in_seed = set(written_seed.loc[written_seed["main_identifier"] == "", "index_identifier"])
in_window = written_coverage.loc[window_start:PANEL_END, "coverage"]
verifications = {
    "the master has one row per FMP symbol in the seed": len(written_master) == len(written_priced),
    "the master holds exactly the seed's symbols": (
        set(written_master["main_identifier"]) == set(written_priced["main_identifier"])
    ),
    "no symbol appears twice in the seed": not bool(
        written_priced["main_identifier"].duplicated().any()
    ),
    "no listing appears twice in the seed": not bool(
        written_seed["index_identifier"].duplicated().any()
    ),
    "the register holds all eleven checks": len(written_issues) == 11,
    "every check has a count": bool(written_issues["count"].notna().all()),
    "files and missing files account for every symbol": (
        len(price_frames) + len(contradicted) + missing_count == len(identifiers)
    ),
    "every weighted listing is a row of the seed": (
        weighted_listings <= set(written_seed["index_identifier"])
    ),
    "every listing with no symbol is named in the register": unpriced_in_seed <= named_unpriced,
    "the window has a first date": pandas.notna(window_start),
    "the window opens no earlier than the floor": window_start >= WINDOW_FLOOR,
    "the window file says the same": (
        pandas.read_csv(WINDOW_PATH)["window_start"].iloc[0] == window_start.date().isoformat()
    ),
    "coverage holds the owner's share on every window date": bool(
        (in_window >= COVERAGE_REQUIRED).all()
    ),
}
failed = [
    name
    for name, passed in verifications.items()
    if not passed
]

if len(failed) > 0:
    message = f"the universe failed verification: {'; '.join(failed)}"

    raise AssertionError(message)

print(f"verified: {len(verifications)} checks on the master, the register and the coverage")